# **Partie 2: Assistant RAG (Génération)**

**Objectif** : Intégrer un LLM pour générer des réponses à partir du contexte retrouvé.

**Choix du LLM:** **Qwen2.5 7B** <br />
**Justificatif** : Qwen2.5 7B offre le meilleur compromis performance/fidélité pour du RAG bilingue.

**Étapes:**

1. Installation/Chargement du modèle.
2. Fonction de génération réponse (contexte + question).
3. Test des 12 scénarios (zero-shot, few-shot, CoT).
4. Évaluation qualitative.
5. Documentation.

**Comparatif des options de LLMs**

| Modèle | Forces | Limites | Décision |
|--------|--------|---------|----------|
| Qwen2.5 14B | Excellent FR/EN, fidélité RAG, licence Apache 2.0 | Plus lourd (14B) | Écarté |
| **Qwen2.5 7B** | Mêmes qualités, plus léger | Moins performant | **Retenu** |
| Mistral 7B | Bon anglais, licence ouverte | FR moins bon, fidélité RAG | Écarté |

## **Chargement du Modèle `Qwen2.5 7B`**

In [1]:
# Installation des librairies nécessaires
# %pip install transformers accelerate torch sentencepiece
# %pip install ipywidgets

## **Imports**

In [2]:
# Imports
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from pathlib import Path
import json
import pandas as pd

In [3]:
# Configuration
PROJET_ROOT = Path.cwd().parent.parent
MODELS_FT_DIR = PROJET_ROOT / "models" / "2-ft"
MODELS_FT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Hugging face login
from huggingface_hub import login
login()

## **Modèle & Tokenizer (+QLoRA)**

In [9]:
# Chargement du modèle et du tokenizer avec optimisation mémoire
model_name = "Qwen/Qwen2.5-7B-Instruct"

print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

Chargement du tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
print("Chargement du modèle en 4-bit (QLoRA) pour réduire l'empreinte mémoire...")

quantization_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
  model_name,
  quantization_config=quantization_config,
  device_map="auto",
  trust_remote_code=True,
)

print("Modèle chargé avec succès.")
model

Chargement du modèle en 4-bit (QLoRA) pour réduire l'empreinte mémoire...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modèle chargé avec succès.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

## **Sauvegarde du modèle & tokenizer**

In [11]:
print("Sauvegarde du modèle et du tokenizer...")

tokenizer.save_pretrained(str(MODELS_FT_DIR / "qwen2.5-7b-instruct-tokenizer"))
# Sauvegarder le modèle (la config, pas les poids 4-bit)
model.save_pretrained(str(MODELS_FT_DIR / "qwen2.5-7b-instruct-model"))

Sauvegarde du modèle et du tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
# Sauvegarder des métadonnées
model_info = {
  "model_name": model_name,
  "quantization": "4-bit (NF4)",
  "device_map": "auto",
  "date_loaded": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(MODELS_FT_DIR / "qwen2.5-7b-instruct-info.json", "w") as f:
  json.dump(model_info, f, indent=2)

print(f"Modèle sauvegardé dans {MODELS_FT_DIR}")

Modèle sauvegardé dans /home/kevin/Documents/Collège LaCité/Cours IA en Informatique (51847)/Hiver 2026/Intelligence Artificielle Générative (IFM31124-0020-H2026)/Projets/ua1/assistant-rag/models/2-ft


## **Génération De Réponse**

In [13]:
# Fonction pour générer une réponse à partir d'un prompt
def generer_reponse(prompt, max_new_tokens=500, temperature=0.1):
  """
  Génère une réponse à partir d'un prompt en utilisant le modèle chargé.
  
  Args:
    prompt (str): Le prompt complet (contexte + question + instructions)
    max_new_tokens (int): Nombre maximum de tokens à générer
    temperature (float): Température pour la génération (0 = déterministe)
  
  Returns:
    str: La réponse générée
  """
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  
  with torch.no_grad():
    outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=temperature,
      do_sample=(temperature > 0),
      pad_token_id=tokenizer.eos_token_id
    )
  
  response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
  return response.strip()

## **Test Rapide de la fonction de Génération**

In [ ]:
# Test rapide de la fonction de génération
test_prompt = "Explique en une phrase ce qu'est un embedding."

print("Test de génération :")
print(f"Prompt : {test_prompt}")
print("-" * 50)
reponse = generer_reponse(test_prompt)
print(f"Réponse : {reponse}")